# Gradient Descent from Scratch

This notebook builds intuition for gradient descent with a one-dimensional function, linear regression, learning-rate comparisons, and a contour plot of the parameter path. It uses only NumPy and Matplotlib, generates deterministic synthetic data, and does not require network access.

Gradient descent minimizes an objective by repeatedly moving parameters opposite the gradient:

$$\theta_{t+1} = \theta_t - \eta\nabla J(\theta_t),$$

where $\theta$ is the parameter vector, $J$ is the objective, and $\eta > 0$ is the learning rate. The gradient points toward the direction of steepest local increase; its negative points toward steepest local decrease.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=4, suppress=True)

## 1. A one-dimensional bowl

Consider the convex function

$$f(x) = (x-3)^2 + 2.$$

Its derivative is $f'(x)=2(x-3)$, and its minimum is at $x=3$. Starting at $x_0=-1$, each update is

$$x_{t+1}=x_t-\eta\,2(x_t-3).$$

The learning rate controls the step size: a small value approaches the minimum slowly; a large value can jump across it repeatedly or diverge.

In [ ]:
def f(x):
    return (x - 3) ** 2 + 2

def grad_f(x):
    return 2 * (x - 3)

learning_rate = 0.15
x_path = [-1.0]

for _ in range(12):
    x = x_path[-1]
    x_path.append(x - learning_rate * grad_f(x))

x_path = np.array(x_path)
print(f"Final x: {x_path[-1]:.6f}; minimum is x = 3")

In [ ]:
x_values = np.linspace(-1.5, 5.5, 300)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x_values, f(x_values), label=r"$f(x)=(x-3)^2+2$", color="navy")
ax.scatter(x_path, f(x_path), c=np.arange(len(x_path)), cmap="viridis", zorder=3)
ax.plot(x_path, f(x_path), color="darkorange", marker="o", markersize=3,
        label="gradient steps")
ax.set(xlabel="x", ylabel="f(x)", title="Gradient descent follows the slope downhill")
ax.legend()
plt.show()

## 2. Fit a line with gradient descent

For observations $(x_i,y_i)$, predict with $\hat y_i=wx_i+b$. We minimize mean squared error:

$$J(w,b)=\frac{1}{n}\sum_{i=1}^{n}(wx_i+b-y_i)^2.$$

Writing $e_i=wx_i+b-y_i$, the partial derivatives are

$$\frac{\partial J}{\partial w}=\frac{2}{n}\sum_i e_i x_i,
\qquad
\frac{\partial J}{\partial b}=\frac{2}{n}\sum_i e_i.$$

We will center and scale the input feature. Feature scaling does not change the underlying prediction problem, but it makes the loss contours better conditioned so one learning rate can make progress in both parameter directions.

In [ ]:
rng = np.random.default_rng(7)
x_raw = np.linspace(0, 10, 60)
y = 2.5 * x_raw + 1.0 + rng.normal(0, 1.2, size=x_raw.shape)

x_mean = x_raw.mean()
x_scale = x_raw.std()
x = (x_raw - x_mean) / x_scale

print(f"Samples: {len(x)}")
print(f"Standardized x: mean={x.mean():.3f}, std={x.std():.3f}")

In [ ]:
def mse_and_gradient(w, b, x, y):
    """Return mean squared error and its gradients for one feature."""
    error = w * x + b - y
    loss = np.mean(error ** 2)
    grad_w = 2 * np.mean(error * x)
    grad_b = 2 * np.mean(error)
    return loss, grad_w, grad_b

def fit_gradient_descent(x, y, learning_rate=0.1, steps=500,
                         initial=(0.0, 0.0)):
    """Optimize slope and intercept; return parameter and loss histories."""
    w, b = initial
    params = [(w, b)]
    losses = []

    for _ in range(steps):
        loss, grad_w, grad_b = mse_and_gradient(w, b, x, y)
        losses.append(loss)
        w -= learning_rate * grad_w
        b -= learning_rate * grad_b
        params.append((w, b))

    final_loss, _, _ = mse_and_gradient(w, b, x, y)
    losses.append(final_loss)
    return np.array(params), np.array(losses)

params, losses = fit_gradient_descent(x, y)
w_scaled, b_fit = params[-1]
w_fit = w_scaled / x_scale
b_original = b_fit - w_scaled * x_mean / x_scale

print(f"Fitted line: y = {w_fit:.3f} x + {b_original:.3f}")
print(f"Final MSE: {losses[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(x_raw, y, alpha=0.75, label="observations")
axes[0].plot(x_raw, w_fit * x_raw + b_original, color="crimson",
             label=f"fit: y={w_fit:.2f}x+{b_original:.2f}")
axes[0].set(xlabel="x", ylabel="y", title="Linear regression fit")
axes[0].legend()

axes[1].plot(losses, color="darkgreen")
axes[1].set(xlabel="step", ylabel="mean squared error",
             title="Training loss")
plt.tight_layout()
plt.show()

## 3. Compare learning rates

The gradient is multiplied by the learning rate at every step. A value that is too small makes progress slowly. A value that is too large can overshoot the minimum and increase the loss instead of reducing it. The stable range depends on the curvature of the objective and the scale of the features.

In [ ]:
rates = {"small (0.005)": 0.005, "good (0.1)": 0.1, "too large (1.1)": 1.1}
fig, ax = plt.subplots(figsize=(8, 4.5))

for label, rate in rates.items():
    _, rate_losses = fit_gradient_descent(x, y, learning_rate=rate, steps=100)
    ax.plot(np.maximum(rate_losses, 1e-12), label=label)

ax.set_yscale("log")
ax.set(xlabel="step", ylabel="MSE (log scale)",
       title="Learning rate changes convergence")
ax.legend()
plt.show()

## 4. See the loss in parameter space

The regression model has two parameters, slope $w$ and intercept $b$. Each point in the $(w,b)$ plane has a loss value. Contour lines connect parameter settings with equal loss; the path shows gradient descent moving across this landscape toward a minimum.

For this squared-error linear regression problem, the objective is convex, so any local minimum is also a global minimum. That guarantee does not automatically apply to arbitrary non-convex models.

In [ ]:
w_grid = np.linspace(w_scaled - 2.0, w_scaled + 2.0, 100)
b_grid = np.linspace(b_fit - 2.0, b_fit + 2.0, 100)
W, B = np.meshgrid(w_grid, b_grid)
loss_grid = np.mean(
    (W[..., None] * x[None, None, :] + B[..., None] - y) ** 2,
    axis=-1,
)

path, _ = fit_gradient_descent(
    x, y, learning_rate=0.1, steps=35,
    initial=(w_scaled - 2.0, b_fit - 2.0)
)

fig, ax = plt.subplots(figsize=(7, 5.5))
contours = ax.contour(W, B, loss_grid, levels=25, cmap="viridis")
ax.clabel(contours, inline=True, fontsize=8)
ax.plot(path[:, 0], path[:, 1], color="crimson", marker="o",
        markersize=3, label="gradient descent path")
ax.scatter([w_scaled], [b_fit], color="gold", edgecolor="black",
           zorder=4, label="fitted parameters")
ax.set(xlabel="scaled-feature slope w", ylabel="intercept b",
       title="MSE contours and optimization path")
ax.legend()
plt.show()

## Takeaways

- Compute the gradient of the objective with respect to every parameter.
- Update parameters in the negative-gradient direction.
- Monitor the objective; a learning rate that is too high can make it diverge.
- Scaling features can make optimization easier by improving the shape of the loss contours.
- Gradient descent is an iterative optimizer, not a guarantee of finding a global optimum for every objective.